In [9]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/content")
DATA_DIR = PROJECT_ROOT
PROCESSED_DIR = PROJECT_ROOT / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", PROJECT_ROOT)
print("CSV files:", [f.name for f in DATA_DIR.glob("*.csv")])

Working directory: /content
CSV files: ['ufc_fights_merged_enhanced.csv']


In [11]:
CSV_PATH = DATA_DIR / "ufc_fights_merged_enhanced.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, low_memory=False)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (8892, 284)


,fight_outcome,outcome,event_date,event_name,event_location,fighter_a_name,fighter_b_name,weight_class,round,time,...,fighter_b_round5_body_strikes_landed,fighter_b_round5_body_strikes_attempted,fighter_b_round5_leg_strikes_landed,fighter_b_round5_leg_strikes_attempted,fighter_b_round5_distance_strikes_landed,fighter_b_round5_distance_strikes_attempted,fighter_b_round5_clinch_strikes_landed,fighter_b_round5_clinch_strikes_attempted,fighter_b_round5_ground_strikes_landed,fighter_b_round5_ground_strikes_attempted
0,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",ROYCE GRACIE,PATRICK SMITH,Open Weight,1.0,1:17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",REMCO PARDOEL,ALBERTA CERRA LEON,Open Weight,1.0,9:51,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",ORLANDO WIET,ROBERT LUCARELLI,Open Weight,1.0,2:50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",SCOTT MORRIS,SEAN DAUGHERTY,Open Weight,1.0,0:20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",PATRICK SMITH,RAY WIZARD,Open Weight,1.0,0:58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

Total rows: 8892
Total columns: 284
1. fight_outcome
2. outcome
3. event_date
4. event_name
5. event_location
6. fighter_a_name
7. fighter_b_name
8. weight_class
9. round
10. time
11. time_format
12. bout_type
13. referee
14. bonus
15. details
16. fighter_a_age
17. fighter_b_age
18. age_diff
19. fighter_a_reach
20. fighter_b_reach
21. reach_diff
22. fighter_a_height
23. fighter_b_height
24. fighter_a_weight
25. fighter_b_weight
26. fighter_a_sig_strikes_landed
27. fighter_b_sig_strikes_landed
28. fighter_a_sig_strikes_attempted
29. fighter_b_sig_strikes_attempted
30. fighter_a_total_strikes_landed
31. fighter_b_total_strikes_landed
32. fighter_a_total_strikes_attempted
33. fighter_b_total_strikes_attempted
34. fighter_a_body_strikes_landed
35. fighter_b_body_strikes_landed
36. fighter_a_body_strikes_attempted
37. fighter_b_body_strikes_attempted
38. fighter_a_clinch_strikes_landed
39. fighter_b_clinch_strikes_landed
40. fighter_a_clinch_strikes_attempted
41. fighter_b_clinch_strikes_at

In [13]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2)
})

missing = missing.sort_values(
    "missing_percent",
    ascending=False
)

display(missing.head(30))

,missing_count,missing_percent
fighter_b_round5_ground_strikes_attempted,2990,33.63
fighter_b_round5_ground_strikes_landed,2990,33.63
fighter_b_round5_clinch_strikes_attempted,2990,33.63
fighter_b_round5_clinch_strikes_landed,2990,33.63
fighter_b_round5_distance_strikes_attempted,2990,33.63
fighter_b_round5_distance_strikes_landed,2990,33.63
fighter_b_round4_clinch_strikes_attempted,2990,33.63
fighter_b_round4_clinch_strikes_landed,2990,33.63
fighter_b_round4_distance_strikes_attempted,2990,33.63
fighter_b_round4_distance_strikes_landed,2990,33.63


In [14]:
print("Exact duplicate rows:", df.duplicated().sum())

if "fight_id" in df.columns:
    print("Unique fight IDs:", df["fight_id"].nunique())
    print("Missing fight IDs:", df["fight_id"].isna().sum())

Exact duplicate rows: 0
Unique fight IDs: 8892
Missing fight IDs: 0


In [15]:
display(
    pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "unique_values": df.nunique(),
        "missing_values": df.isnull().sum()
    })
)

,data_type,unique_values,missing_values
fight_outcome,object,3,0
outcome,int64,2,0
event_date,object,789,0
event_name,object,794,0
event_location,object,175,0
...,...,...,...
fighter_b_round5_distance_strikes_attempted,float64,92,2990
fighter_b_round5_clinch_strikes_landed,float64,14,2990
fighter_b_round5_clinch_strikes_attempted,float64,16,2990
fighter_b_round5_ground_strikes_landed,float64,16,2990


In [16]:
# Analyze columns with high missing percentages
missing_percent = df.isnull().mean() * 100

print("Columns with >95% missing values:",
      (missing_percent > 95).sum())

print("\nColumns with >80% missing values:",
      (missing_percent > 80).sum())

display(
    missing_percent[missing_percent > 80]
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)

Columns with >95% missing values: 0

Columns with >80% missing values: 0


,missing_percent


In [17]:
important_columns = [
    "fight_id",
    "event_id",
    "event_name",
    "event_date",
    "fighter_a_name",
    "fighter_b_name",
    "fight_outcome",
    "method",
    "weight_class",
]

for col in important_columns:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print("Missing:", df[col].isna().sum())
        print("Unique:", df[col].nunique())
        print(df[col].value_counts(dropna=False).head(5))
    else:
        print(f"\nMissing column from schema: {col}")


--- fight_id ---
Missing: 0
Unique: 8892
fight_id
fb6b38c62d1a78be    1
00835554f95fa911    1
17ee4caf06981b81    1
3b020d4914b44fc8    1
4acab67848e78327    1
Name: count, dtype: int64

--- event_id ---
Missing: 0
Unique: 794
event_id
ufc-283-teixeira-vs-hill              16
ufc-259-blachowicz-vs-adesanya        15
ufc-fight-night-whittaker-vs-till     15
ufc-fight-night-aspinall-vs-tybura    15
ufc-286-edwards-vs-usman-3            15
Name: count, dtype: int64

--- event_name ---
Missing: 0
Unique: 794
event_name
UFC 283: Teixeira vs. Hill              16
UFC 259: Blachowicz vs. Adesanya        15
UFC Fight Night: Whittaker vs. Till     15
UFC Fight Night: Aspinall vs. Tybura    15
UFC 286: Edwards vs. Usman 3            15
Name: count, dtype: int64

--- event_date ---
Missing: 0
Unique: 789
event_date
2016-11-19    25
2014-10-04    22
2014-05-31    22
2014-08-23    21
2014-06-28    21
Name: count, dtype: int64

--- fighter_a_name ---
Missing: 0
Unique: 2335
fighter_a_name
JIM MILLE

In [18]:
df_clean = df.copy()

# Normalize whitespace in text columns without changing missing values
text_columns = df_clean.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    df_clean[col] = df_clean[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Original shape:", df.shape)
print("Working copy shape:", df_clean.shape)

Original shape: (8892, 284)
Working copy shape: (8892, 284)


In [19]:
df_clean = df.copy()

# Remove leading and trailing spaces from text values
text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df_clean[col] = df_clean[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Original shape:", df.shape)
print("Working copy shape:", df_clean.shape)
print("Exact duplicate rows:", df_clean.duplicated().sum())

Original shape: (8892, 284)
Working copy shape: (8892, 284)
Exact duplicate rows: 0


In [20]:
if "fight_id" in df_clean.columns:
    print("Total rows:", len(df_clean))
    print("Unique fight IDs:", df_clean["fight_id"].nunique())
    print("Missing fight IDs:", df_clean["fight_id"].isna().sum())
    print("Duplicate fight IDs:", df_clean["fight_id"].duplicated().sum())

Total rows: 8892
Unique fight IDs: 8892
Missing fight IDs: 0
Duplicate fight IDs: 0


In [21]:
from pathlib import Path

PROCESSED_DIR = Path("/content/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV_PATH = PROCESSED_DIR / "ufc_fights_clean.csv"

df_clean.to_csv(CLEAN_CSV_PATH, index=False)

print("Saved:", CLEAN_CSV_PATH)
print("File size (MB):", round(CLEAN_CSV_PATH.stat().st_size / (1024 * 1024), 2))

Saved: /content/processed/ufc_fights_clean.csv
File size (MB): 11.12


In [22]:
print("Fight outcome counts:")
display(
    df_clean["fight_outcome"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("Unique outcome values:")
print(df_clean["fight_outcome"].unique())

Fight outcome counts:


,count
fight_outcome,
red_win,5610
blue_win,3281
draw,1


Unique outcome values:
['red_win' 'blue_win' 'draw']


In [23]:
print("Missing event dates:", df_clean["event_date"].isna().sum())
print("Sample dates:")
display(df_clean["event_date"].head(10))

Missing event dates: 0
Sample dates:


,event_date
0,1994-03-11
1,1994-03-11
2,1994-03-11
3,1994-03-11
4,1994-03-11
5,1994-03-11
6,1994-03-11
7,1994-03-11
8,1994-03-11
9,1994-03-11


In [24]:
from pathlib import Path

PROCESSED_DIR = Path("/content/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV_PATH = PROCESSED_DIR / "ufc_fights_clean.csv"

df_clean.to_csv(CLEAN_CSV_PATH, index=False)

print("Saved:", CLEAN_CSV_PATH)
print("Rows and columns:", df_clean.shape)

Saved: /content/processed/ufc_fights_clean.csv
Rows and columns: (8892, 284)


In [25]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [26]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/RAG UFC-AI-Analyst")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV_PATH = PROCESSED_DIR / "ufc_fights_clean.csv"

df_clean.to_csv(CLEAN_CSV_PATH, index=False)

print("Saved to:", CLEAN_CSV_PATH)
print("Dataset shape:", df_clean.shape)
print("File size (MB):", round(CLEAN_CSV_PATH.stat().st_size / (1024 * 1024), 2))

Saved to: /content/drive/MyDrive/RAG UFC-AI-Analyst/data/processed/ufc_fights_clean.csv
Dataset shape: (8892, 284)
File size (MB): 11.12


In [27]:
import pandas as pd

check_df = pd.read_csv(CLEAN_CSV_PATH, low_memory=False)

print("Saved dataset shape:", check_df.shape)
print("Missing fight IDs:", check_df["fight_id"].isna().sum())
print("Duplicate fight IDs:", check_df["fight_id"].duplicated().sum())

Saved dataset shape: (8892, 284)
Missing fight IDs: 0
Duplicate fight IDs: 0
